In [ ]:
import numpy as np
import pandas as pd
import regex as re
import json
import tweepy
from transformers import AutoTokenizer
import os
import random
import nltk
import subprocess, requests, zipfile, io
import datetime

save_path = "../data/"

### Loading tweets

In [ ]:
# tweet ids are available but need to be hydrated with the Twitter API

with open(save_path + "cctweets_ids.txt", "r") as f:
    tweet_ids = [id_[:-1] for id_ in f.readlines()]
    
print(len(tweet_ids))

In [ ]:
# insert access tokens from a Twitter developer account
consumer_key = ""
consumer_secret = ""
access_token = ""
access_token_secret = ""

auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)
api = tweepy.API(auth, wait_on_rate_limit = True)


In [ ]:
tweets = {}

def retrieve_full_text(tweet):
    if "extended_tweet" in tweet:
        text = tweet['extended_tweet']['full_text']
    elif "full_text" in tweet:
        text = tweet['full_text']
    elif "text" in tweet:
            text = tweet['text']
    return text

for i in range(0, len(tweet_ids), 100):

    hydrated = api.lookup_statuses(tweet_ids[i:i+100], include_entities = False, trim_user = True, 
                                   tweet_mode = "extended")
    
    hydrated = {str(tweet._json["id"]): {"text": retrieve_full_text(tweet._json),
                                    "created_at": tweet._json["created_at"]} for tweet in hydrated}
    
    tweets.update(hydrated)
    
print(len(tweets))


### Fix Encoding and remove URLs

In [ ]:
def fix_encoding(text):
    
    text = text.replace('&amp;', '&')
    text = text.replace('&lt;', '<')
    text = text.replace('&gt;', '>')
    
    return text

def remove_urls(text):
    
    text = re.sub("http\S+|ttps\S+", "", text)
    #ttps because there is one widely shared instance where the h was forgotten
    text = re.sub("\S+html", "", text)
    
    return text


for tweet_id in list(tweets.keys()):
    
    text = tweets[tweet_id]["text"]

    
    text = fix_encoding(text)
    text = remove_urls(text)
    
    tweets[tweet_id]["text"] = text
    
    # removing a hundred tweets that are badly encoded
    if re.search("%[0-9]", text):
        
        del tweets[tweet_id]
    
    
print(len(tweets))

### Removing tweets with mentions and hashtags only

In [ ]:
mention_pattern = re.compile("(@\S+ ){2,}", re.IGNORECASE)
hashtag_pattern = re.compile("(#\S+ ){2,}", re.IGNORECASE)
activist_pattern = re.compile(r"[@#]?greta[-_ ]?[@#]?thunberg|[@#]?greta|[@#]?thunberg", re.IGNORECASE)

mention_hashtag_tweets = {}

for tweet_id in list(tweets.keys()):
    
    text = tweets[tweet_id]["text"]
        
    text = re.sub(mention_pattern, "", text) if mention_pattern.search(text) else text
    text = re.sub(hashtag_pattern, "", text) if hashtag_pattern.search(text) else text
            
    if activist_pattern.search(text):
            
        tweets[tweet_id]["text"] = text
            
    else:
        
        mention_hashtag_tweets.update({tweet_id:tweets[tweet_id]})
        del tweets[tweet_id]
        
print(len(tweets))

### Standardizing and removing leading/trailing whitespace

In [ ]:
for tweet_id in list(tweets.keys()):
    
    text = tweets[tweet_id]["text"]
    
    text = re.sub(r"\s+", " ", text).strip()

    tweets[tweet_id]["text"] = text

### Pruning repeating patterns (both single characters and whole words)

In [ ]:
# repetitions > 3 are replaced by three repetitions

for tweet_id in list(tweets.keys()):
    
    text = tweets[tweet_id]["text"]
    
    text = re.sub(r'(.+?)(\1){3,}', r'\1'*3, text)
    
    tweets[tweet_id]["text"] = text


### Sorting tweets and removing duplicates

In [ ]:
df = pd.DataFrame.from_dict(tweets, orient = "index")
df.sort_index(inplace = True)
df.drop_duplicates(subset="text", keep ="first", inplace = True)
tweets = df.to_dict(orient = "index")

print(len(tweets))

### Masking

In [ ]:
def masking(text, dataset_name):
    
    if dataset_name == "cctweets-random":
    
        tokens = text.split(" ")
        i = random.choice(range(0, len(tokens)))
        eos = tokens[i][-1] if tokens[i][-1] in [".", ",", ":", "?", "!", ";"] else ""
        tokens[i] = "".join(("[MASK]", eos))
        text_masked = " ".join(tokens)
        

    elif dataset_name == "cctweets-activist":
        
        activist_pattern = re.compile(r"[@#]?greta[-_ ]?[@#]?thunberg|[@#]?greta|[@#]?thunberg", re.IGNORECASE)
        text_masked = re.sub(activist_pattern, "[MASK]", text)
    
    return text_masked
    
    

In [ ]:
dataset_dict = {"cctweets-random": {}, "cctweets-activist": {}}

random.seed(42)

sentence_pattern = re.compile("\[MASK\] [a-z]|[a-z] \[MASK\]")

for tweet_id, tweet in list(tweets.items()):
    
    text = tweet["text"]
    
    for dataset_name in dataset_dict.keys():
        

        text_masked = masking(text, dataset_name)
        
        
        sentences = [sentence for sentence in nltk.sent_tokenize(text_masked) if sentence_pattern.search(sentence)]
        
        if len(sentences)  == 0:
            
            continue
            
        else:
            text_masked = sentences[0] if isinstance(sentences, list) else sentences
            
            if len(re.findall("\[MASK\]", text_masked)) > 1:
                
                continue
        
        
        tokens = text_masked.split(" ")
        n_tokens = len(tokens)
        n_characters = len(text_masked)
        
        if n_tokens <3 or n_characters> 255:
            
            continue
        
        match = re.search("\[MASK\]", text_masked)
        position = match.span()[0]
        relative_position = round(position/len(text_masked), 2)
        first_token = True if position == 0 else False
        last_token = True if position == n_characters-6 else False
        eof = True if text_masked[position:position+7] == "[MASK]." else False
        
       
        
        dataset_dict[dataset_name][tweet_id] = {
            "text" : text,
            "text_masked" : text_masked,
            "position" : position,
            "relative_position" : relative_position,
            "n_tokens": n_tokens,
            "n_characters" :n_characters,
            "first-token": first_token,
            "last-token": last_token,
            "end-of-sentence": eof,
            "created_at": tweet["created_at"]

        }
        
    

## Creating dataframes

In [ ]:
dataset_dfs = {}

for dataset_name in dataset_dict.keys():
    
    df = pd.DataFrame.from_dict(dataset_dict[dataset_name], orient = "index")
    df.sort_index(inplace = True)
    df.drop_duplicates(subset="text", keep ="first", inplace = True)
    
    print(f"{dataset_name}: {len(df)}")
    
    dataset_dfs[dataset_name] = df
    

## Saving

In [ ]:
for dataset_name in dataset_dict.keys():
    
    df = pd.DataFrame.from_dict(dataset_dict[dataset_name], orient = "index").reset_index()
    df.rename(columns = {"index": "id"}, inplace = True)
    
    df.to_csv(save_path + dataset_name + "_df.csv", encoding = "utf8")
    
    with open(save_path + dataset_name + "_texts.txt", "w",encoding = "utf8")as f:
        
        [f.write(text + "\n") for text in df["text_masked"]]


### Verifying that no tweet will exceed 256 tokens for any model

In [ ]:
for dataset_name in dataset_dfs:
    
    print(dataset_name)
    df = dataset_dfs[dataset_name]

    long_texts = list(df.loc[(df["n_tokens"] > 50) | (df["n_characters"]>200), "text"])

    models = ['bert-base-uncased', 'roberta-base', 'bert-base-cased', 'distilbert-base-uncased', 
            'albert-base-v2', 'google/electra-base-discriminator']

    for model in models:
    
        tokenizer = AutoTokenizer.from_pretrained(model)
        max_len = tokenizer(long_texts, return_tensors = "pt", padding = True, truncation = True)["input_ids"].shape[1]
        print(f'{model} max_len = {max_len}')      
    print("-"*100)

# Overview

In [ ]:
for dataset_name in dataset_dfs:
    
    print(dataset_name)
    df = dataset_dfs[dataset_name]

    # plotting over time
    dates = list(df["created_at"].values)
    dates_parsed = [datetime.datetime.strptime(date, '%a %b %d %H:%M:%S %z %Y') for date in dates]
    dates_parsed.sort()
    delta  = dates_parsed[-1] - dates_parsed[0]
    hours = delta.total_seconds()/3600
    nhours = round(hours)
    series = pd.Series(dates_parsed)
    series.hist(figsize = [10, 7], bins = nhours)
    
    # summary
    summary ={"tweets": len(df),
     "first_day": min(dates_parsed).strftime("%d.%m.%Y"),
     "last_day": max(dates_parsed).strftime("%d.%m.%Y")}
 
    print(summary)